In [18]:
import pandas as pd

cpu = pd.read_csv("../raw_data/cpu_raw.csv")

cpu.head()

,cit,year,CPU_US,CPU_UK,CPU_CAN,CPU_AUS,CPU_GER,CPU_ITA,CPU_ESP,CPU_MEX,...,CPU_NZ,CPU_FRA,CPU_pos_US,CPU_pos_UK,CPU_pos_CAN,CPU_pos_AUS,CPU_neg_US,CPU_neg_UK,CPU_neg_CAN,CPU_neg_AUS
0,"Please cite as follows: Basaglia, P, C Beresty...",1990,35.505461,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,62.949975,NaN,37.646823,NaN,NaN,NaN,26.988389,NaN,NaN,NaN
1,NaN,1991,53.241677,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,46.103458,NaN,123.131118,NaN,NaN,NaN,37.713738,NaN,NaN,NaN
2,NaN,1992,63.808044,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,105.675123,NaN,81.496635,NaN,NaN,NaN,71.525520,NaN,NaN,NaN
3,NaN,1993,63.324147,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,29.358530,NaN,97.789189,NaN,NaN,NaN,47.328534,NaN,NaN,NaN
4,NaN,1994,49.083190,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,42.510757,NaN,32.469914,NaN,NaN,NaN,30.741161,NaN,NaN,NaN


In [19]:
print(cpu.columns.tolist())
print(cpu.shape)
print(cpu["year"].min(), cpu["year"].max())

['cit', 'year', 'CPU_US', 'CPU_UK', 'CPU_CAN', 'CPU_AUS', 'CPU_GER', 'CPU_ITA', 'CPU_ESP', 'CPU_MEX', 'CPU_IRL', 'CPU_NZ', 'CPU_FRA', 'CPU_pos_US', 'CPU_pos_UK', 'CPU_pos_CAN', 'CPU_pos_AUS', 'CPU_neg_US', 'CPU_neg_UK', 'CPU_neg_CAN', 'CPU_neg_AUS']
(36, 21)
1990 2025


In [20]:
# 기본 CPU 지수 열만 선택
main_cpu_cols = [
    col for col in cpu.columns
    if col.startswith("CPU_")
    and not col.startswith("CPU_pos_")
    and not col.startswith("CPU_neg_")
]

# year와 기본 CPU 열만 남기고 연구기간 필터링
cpu_main = cpu[["year"] + main_cpu_cols].copy()
cpu_main = cpu_main[
    (cpu_main["year"] >= 2000) &
    (cpu_main["year"] <= 2019)
]

# wide format → long format
cpu_long = cpu_main.melt(
    id_vars="year",
    var_name="country_raw",
    value_name="CPU"
)

# CPU_US → US 형태로 국가명 정리
cpu_long["country_raw"] = cpu_long["country_raw"].str.replace(
    "CPU_", "", regex=False
)

# ISO3 국가코드로 변환
country_map = {
    "US": "USA",
    "UK": "GBR",
    "CAN": "CAN",
    "AUS": "AUS",
    "GER": "DEU",
    "ITA": "ITA",
    "ESP": "ESP",
    "MEX": "MEX",
    "IRL": "IRL",
    "NZ": "NZL",
    "FRA": "FRA"
}

cpu_long["country_code"] = cpu_long["country_raw"].map(country_map)

# 보기 좋은 순서로 정리
cpu_long = cpu_long[
    ["country_code", "year", "CPU"]
].sort_values(
    ["country_code", "year"]
).reset_index(drop=True)

cpu_long.head(10)

,country_code,year,CPU
0,AUS,2000,37.986378
1,AUS,2001,32.302196
2,AUS,2002,23.147430
3,AUS,2003,32.739329
4,AUS,2004,32.399758
5,AUS,2005,27.293299
6,AUS,2006,42.008283
7,AUS,2007,92.353258
8,AUS,2008,129.333148
9,AUS,2009,113.746408


In [21]:
cpu_long.to_csv(
    "../processed_data/cpu_long_2000_2019.csv",
    index=False
)

In [22]:
coverage = cpu_long.groupby("country_code")["CPU"].agg(
    total_years="size",
    available_years="count",
    missing_values=lambda x: x.isna().sum()
)

coverage

,total_years,available_years,missing_values
country_code,,,
AUS,20,20,0
CAN,20,16,4
DEU,20,19,1
ESP,20,20,0
FRA,20,19,1
GBR,20,20,0
IRL,20,17,3
ITA,20,20,0
MEX,20,16,4


In [23]:
cpu_long.to_csv(
    "../processed_data/cpu_long_2000_2019.csv",
    index=False
)

In [24]:
import pandas as pd

cpu_long = pd.read_csv(
    "../processed_data/cpu_long_2000_2019.csv"
)

cpu_long.head()

,country_code,year,CPU
0,AUS,2000,37.986378
1,AUS,2001,32.302196
2,AUS,2002,23.147430
3,AUS,2003,32.739329
4,AUS,2004,32.399758


In [25]:
coverage = cpu_long.groupby("country_code")["CPU"].agg(
    total_years="size",
    available_years="count"
)

coverage["missing_values"] = (
    coverage["total_years"] - coverage["available_years"]
)

coverage

,total_years,available_years,missing_values
country_code,,,
AUS,20,20,0
CAN,20,16,4
DEU,20,19,1
ESP,20,20,0
FRA,20,19,1
GBR,20,20,0
IRL,20,17,3
ITA,20,20,0
MEX,20,16,4


In [26]:
cpu_long.loc[
    cpu_long["CPU"].isna(),
    ["country_code", "year"]
]

,country_code,year
20,CAN,2000
21,CAN,2001
22,CAN,2002
23,CAN,2003
40,DEU,2000
80,FRA,2000
120,IRL,2000
121,IRL,2001
122,IRL,2002
160,MEX,2000


In [27]:
coverage.to_csv(
    "../processed_data/cpu_coverage.csv"
)